In [ ]:
# hide
import numpy as np
import pyquist as pq


def iter_frames(audio, hop_length, frame_length):
    x = np.asarray(audio.samples).reshape(-1)
    for start in range(0, len(x), hop_length):
        yield x[start:start + frame_length]


def overlap_add(frames, hop_length, sample_rate):
    frames = [f for f in frames]
    frame_length = len(frames[0])
    out = np.zeros(hop_length * (len(frames) - 1) + frame_length)
    for k, frame in enumerate(frames):
        if len(frame) == frame_length:
            out[k * hop_length:k * hop_length + frame_length] += frame
    return pq.Audio(out.astype(np.float32), sample_rate)


def hann(n):
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / n))

In [ ]:
# Extract frames and glue them back together. With a rectangular window and
# N_H = N_F (0% overlap) this is perfect reconstruction. Try N_H = N_F // 2
# (overlap, doubles the amplitude) or N_H = 2 * N_F (gaps) and listen!
audio = pq.Audio.from_file("../assets/audio-trio.wav")
N_F = 1024        # frame length (samples)
N_H = 1024        # hop length  (samples)

frames = iter_frames(audio, N_H, N_F)
reconstructed = overlap_add(frames, N_H, audio.sample_rate)
pq.play(reconstructed)